# 05 — Báo cáo kinh doanh trên tập kiểm định cuối đã khóa

Họ mô hình, bộ đặc trưng, số cụm và logic đặt tên đã được khóa trước notebook này. Mô hình hiện được huấn luyện lại đến hết ngày 31 tháng 10 năm 2010; tập kiểm định cuối từ **1 tháng 11 đến 9 tháng 12** chỉ được mở đúng một lần.


In [1]:
from pathlib import Path
import json, os, warnings
os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".matplotlib"))
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "2")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="colorblind")
pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_PATH = ROOT / "data" / "online_retail_II.csv"
OUT = ROOT / "outputs"
INTERMEDIATE = OUT / "intermediate"
REPORTS = OUT / "reports"
FIGURES = OUT / "figures"
MODELS = OUT / "models"
for directory in (INTERMEDIATE, REPORTS, FIGURES, MODELS):
    directory.mkdir(parents=True, exist_ok=True)
(ROOT / "experiments").mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
DEV_CUTOFF = pd.Timestamp("2010-08-31 23:59:59")
VALIDATION_END = pd.Timestamp("2010-10-31 23:59:59")
HOLDOUT_END = pd.Timestamp("2010-12-09 23:59:59")
print(f"Thư mục gốc dự án: {ROOT}")


def build_customer_features(transactions, cutoff):
    d = transactions.loc[transactions["InvoiceDate"] <= cutoff].copy()
    invoice = (d.groupby(["Customer ID", "Invoice"], as_index=False)
                 .agg(InvoiceDate=("InvoiceDate", "min"),
                      OrderValue=("LineRevenue", "sum"),
                      BasketQuantity=("Quantity", "sum"),
                      BasketProducts=("StockCode", "nunique")))
    base = invoice.groupby("Customer ID").agg(
        LastPurchase=("InvoiceDate", "max"),
        FirstPurchase=("InvoiceDate", "min"),
        Frequency=("Invoice", "nunique"),
        Monetary=("OrderValue", "sum"),
        AOV=("OrderValue", "mean"),
        AvgBasketSize=("BasketQuantity", "mean"),
    )
    base["Recency"] = (cutoff.normalize() + pd.Timedelta(days=1) - base["LastPurchase"].dt.normalize()).dt.days
    base["ActiveDays"] = (base["LastPurchase"].dt.normalize() - base["FirstPurchase"].dt.normalize()).dt.days
    base["IsRepeat"] = (base["Frequency"] > 1).astype(int)
    diversity = d.groupby("Customer ID")["StockCode"].nunique().rename("ProductDiversity")
    base = base.join(diversity)
    base["PurchaseCadence"] = np.where(
        base["IsRepeat"].eq(1), base["ActiveDays"] / (base["Frequency"] - 1), np.nan
    )
    return base.drop(columns=["LastPurchase", "FirstPurchase"]).sort_index()


def future_outcomes(transactions, start, end, eligible_customers):
    future = transactions.loc[(transactions["InvoiceDate"] > start) & (transactions["InvoiceDate"] <= end)].copy()
    outcome = future.groupby("Customer ID").agg(
        FutureRevenue=("LineRevenue", "sum"),
        FutureOrders=("Invoice", "nunique"),
    )
    result = pd.DataFrame(index=pd.Index(eligible_customers, name="Customer ID")).join(outcome).fillna(0)
    result["FutureActive"] = (result["FutureOrders"] > 0).astype(int)
    return result

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score
import joblib, hashlib

Thư mục gốc dự án: C:\Users\lqb46\Documents\Projects\Clustomer


In [2]:
tx=pd.read_csv(INTERMEDIATE/"uk_completed_purchases.csv",parse_dates=["InvoiceDate"])
final_features=pd.read_csv(INTERMEDIATE/"final_train_features.csv",index_col="Customer ID")
selection=json.loads((REPORTS/"locked_selection.json").read_text(encoding="utf-8"))
name_map={int(k):v for k,v in json.loads((REPORTS/"segment_names.json").read_text(encoding="utf-8")).items()}
cols=selection["features"]
development=joblib.load(MODELS/"development_model.joblib")
scaler=StandardScaler().fit(np.log1p(final_features[cols])); Z=scaler.transform(np.log1p(final_features[cols]))
k=int(selection["config"].split('=')[1])
if selection["model"]=="kmeans":
    # Chuyển tâm cụm phát triển sang không gian của bộ chuẩn hóa cuối. Cách này giữ
    # nguyên ý nghĩa cụm/tên trong khi vẫn cập nhật tâm cụm bằng dữ liệu tháng 9–10.
    raw_log_centres=development["scaler"].inverse_transform(development["model"].cluster_centers_)
    aligned_init=scaler.transform(raw_log_centres)
    model=KMeans(n_clusters=k,init=aligned_init,n_init=1,random_state=RANDOM_STATE).fit(Z)
    labels=model.labels_; uncertainty=model.transform(Z).min(axis=1)
elif selection["model"]=="gmm": model=GaussianMixture(n_components=k,reg_covar=1e-5,random_state=RANDOM_STATE).fit(Z); labels=model.predict(Z); uncertainty=1-model.predict_proba(Z).max(axis=1)
else: raise ValueError("Suy luận triển khai cần một mô hình có khả năng dự đoán; mô hình đã khóa không được hỗ trợ")
holdout=future_outcomes(tx,VALIDATION_END,HOLDOUT_END,final_features.index)
print(f"Khách hàng huấn luyện cuối: {len(final_features):,}; số ngày của tập kiểm định cuối: {(HOLDOUT_END-VALIDATION_END).days}")

Khách hàng huấn luyện cuối: 3,624; số ngày của tập kiểm định cuối: 39


## 1. Hiệu quả cuối cùng của các phân khúc

In [3]:
scored=final_features.copy(); scored["Cluster"]=labels; scored["SegmentName"]=scored.Cluster.map(name_map); scored["AssignmentUncertainty"]=uncertainty
threshold=float(np.quantile(uncertainty,.95)); scored["ManualReview"]=scored.AssignmentUncertainty>=threshold
final_profile=scored.join(holdout).groupby(["Cluster","SegmentName"]).agg(
    Customers=("Frequency","size"),RecencyMedian=("Recency","median"),FrequencyMedian=("Frequency","median"),
    MonetaryMedian=("Monetary","median"),HoldoutRevenueMean=("FutureRevenue","mean"),
    HoldoutActiveRate=("FutureActive","mean"),HoldoutOrdersMean=("FutureOrders","mean"),
    ManualReviewRate=("ManualReview","mean"))
final_profile["CustomerShare"]=final_profile.Customers/len(scored)
display(final_profile)

,,Customers,RecencyMedian,FrequencyMedian,MonetaryMedian,HoldoutRevenueMean,HoldoutActiveRate,HoldoutOrdersMean,ManualReviewRate,CustomerShare
Cluster,SegmentName,,,,,,,,,
0,Needs Attention,741,102.000,3.000,867.600,169.386,0.339,0.475,0.024,0.204
1,Champions,263,12.000,14.000,"5,983.500","2,084.229",0.844,3.266,0.202,0.073
2,Loyal Growth,816,22.000,5.000,"1,833.270",336.386,0.550,0.982,0.044,0.225
3,Dormant,1041,188.000,1.000,216.410,61.523,0.187,0.240,0.028,0.287
4,New & Promising,763,19.000,1.000,406.020,122.212,0.304,0.489,0.060,0.211


In [4]:
def eta_squared(values, labels):
    f=pd.DataFrame({"y":np.log1p(values),"label":labels}); overall=f.y.mean(); total=((f.y-overall)**2).sum()
    return float(sum(len(g)*(g.y.mean()-overall)**2 for _,g in f.groupby('label'))/total) if total else 0.0
metrics={"silhouette":float(silhouette_score(Z,labels)),"davies_bouldin":float(davies_bouldin_score(Z,labels)),
         "min_segment_share":float(pd.Series(labels).value_counts(normalize=True).min()),
         "holdout_revenue_eta2":eta_squared(holdout.FutureRevenue,labels),
         "holdout_active_eta2":eta_squared(holdout.FutureActive,labels),
         "holdout_active_rate":float(holdout.FutureActive.mean()),"holdout_revenue":float(holdout.FutureRevenue.sum()),
         "manual_review_threshold":threshold}
print(json.dumps(metrics,indent=2))
assert metrics["min_segment_share"] >= 0.05, "Không đạt ngưỡng khả năng hành động trên tập kiểm định cuối"
assert metrics["holdout_revenue_eta2"] >= 0.10, "Khả năng phân tách doanh thu trên tập kiểm định cuối quá yếu"
assert metrics["holdout_active_eta2"] >= 0.05, "Khả năng phân tách hoạt động trên tập kiểm định cuối quá yếu"

{
  "silhouette": 0.33670838578395235,
  "davies_bouldin": 0.9667360857063244,
  "min_segment_share": 0.07257174392935982,
  "holdout_revenue_eta2": 0.19989886427911835,
  "holdout_active_eta2": 0.14688674046542613,
  "holdout_active_rate": 0.37224061810154524,
  "holdout_revenue": 1105450.352,
  "manual_review_threshold": 1.5053159577917596
}


## 2. Ma trận hành động kinh doanh

In [5]:
actions={"Champions":"Duy trì trải nghiệm; thử nghiệm giới thiệu và quyền tiếp cận sớm.","Loyal Growth":"Bán chéo phù hợp, đồng thời kiểm soát biên lợi nhuận.",
         "New & Promising":"Gửi lời nhắc mua lại đúng thời điểm và đo lường hiệu quả gia tăng.","Needs Attention":"Tái kích hoạt có chọn lọc và giới hạn chi phí ưu đãi.",
         "Dormant":"Ưu tiên kênh chi phí thấp hoặc không liên hệ nếu chưa chứng minh được hiệu quả gia tăng."}
action_table=final_profile.reset_index()[["SegmentName","Customers","CustomerShare","HoldoutActiveRate","HoldoutRevenueMean"]]
action_table["RecommendedExperiment"]=action_table.SegmentName.map(actions)
display(action_table)

,SegmentName,Customers,CustomerShare,HoldoutActiveRate,HoldoutRevenueMean,RecommendedExperiment
0,Needs Attention,741,0.204,0.339,169.386,Tái kích hoạt có chọn lọc và giới hạn chi phí ...
1,Champions,263,0.073,0.844,"2,084.229",Duy trì trải nghiệm; thử nghiệm giới thiệu và ...
2,Loyal Growth,816,0.225,0.550,336.386,"Bán chéo phù hợp, đồng thời kiểm soát biên lợi..."
3,Dormant,1041,0.287,0.187,61.523,Ưu tiên kênh chi phí thấp hoặc không liên hệ n...
4,New & Promising,763,0.211,0.304,122.212,Gửi lời nhắc mua lại đúng thời điểm và đo lườn...


## 3. Artifact, metadata quản trị và đầu ra có thể tái lập

In [6]:
artifact={"model":model,"scaler":scaler,"features":cols,"segment_names":name_map,
          "uncertainty_threshold":threshold,"population":"Khách hàng đã định danh tại Vương quốc Anh với giao dịch mua hoàn tất và giá trị dương",
          "trained_through":str(VALIDATION_END),"trained_at":pd.Timestamp.utcnow().isoformat(),
          "metrics":metrics,"model_version":"1.0.0",
          "reference_statistics":{column:{"mean":float(final_features[column].mean()),
            "std":float(final_features[column].std()),
            "quantiles":[float(v) for v in final_features[column].quantile([0,.25,.5,.75,1])]}
            for column in cols}}
artifact_path=MODELS/"clustomer_segmenter.joblib"; joblib.dump(artifact,artifact_path)
checksum=hashlib.sha256(artifact_path.read_bytes()).hexdigest()
metadata={"model_version":"1.0.0","trained_through":str(VALIDATION_END),"data_end":str(HOLDOUT_END),
          "selection":selection,"holdout_metrics":metrics,"artifact_sha256":checksum,
          "limitations":["Chỉ gồm khách hàng đã định danh tại Vương quốc Anh","Dữ liệu lịch sử chỉ bao phủ một năm","Phân khúc không ước lượng tác động nhân quả của chiến dịch","5% lượt gán bất định nhất cần được rà soát"]}
scored.join(holdout).to_csv(OUT/"customer_segments.csv")
final_profile.to_csv(REPORTS/"final_segment_profiles.csv")
(REPORTS/"production_report.json").write_text(json.dumps(metadata,indent=2,ensure_ascii=False),encoding="utf-8")
print(f"Đã lưu {artifact_path.name}; SHA256={checksum}")

Đã lưu clustomer_segmenter.joblib; SHA256=4c2b1727be9abd8ceadf38af79b189385b380def001edcca5aa8115c5f6d48bb


## Khuyến nghị triển khai và các giới hạn

Chỉ triển khai như một công cụ hỗ trợ ra quyết định CRM nếu bảng kiểm định cuối cho thấy khác biệt đáng kể về hoạt động hoặc giá trị tương lai và không phân khúc nào vi phạm ngưỡng khả thi 5%. Mô hình **không** suy luận sở thích, mức độ phản hồi chiến dịch hay điều kiện tiếp cận của khách hàng. Cần theo dõi lược đồ đầu vào, độ trôi đặc trưng, biến động quy mô phân khúc, độ bất định và kết quả chiến dịch thực tế. Chỉ huấn luyện lại khi độ trôi kéo dài hoặc định nghĩa kinh doanh thay đổi, không đơn thuần theo một lịch cố định.

5% lượt gán có độ bất định cao nhất được đánh dấu để rà soát. Khách hàng mới chưa có lịch sử giao dịch và khách hàng ngoài Vương quốc Anh nằm ngoài phạm vi, thay vì bị ép vào một phân khúc.
